In [45]:
#IMPORTAÇÕES
import numpy as np
import pandas as pd
#%matplotlib inline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.arima_model import ARMA,ARMAResults,ARIMA,ARIMAResults
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
#from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error
from statsmodels.tools.eval_measures import rmse
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

In [46]:
print("=== PROCESSANDO DADOS DO EXCEL ===")
# Lendo os dados do Excel (código original adaptado)
df_excel = pd.read_excel(
    "../Controle_Faturamento_2023.xlsx",
    sheet_name="Consumo Médio - Clientes",
)

=== PROCESSANDO DADOS DO EXCEL ===


In [47]:
df_excel

,Cliente,Nº Cliente,UC,2019-06-01 00:00:00,2019-07-01 00:00:00,2019-08-01 00:00:00,2019-09-01 00:00:00,2019-10-01 00:00:00,2019-11-01 00:00:00,2019-12-01 00:00:00,...,Unnamed: 58,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67
0,Lucile Confecções LTDA,7.005714e+09,3003858507,1246.0,1081.0,1184.0,1549.0,2045.0,4017.0,2386.0,...,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez
1,CLL Hotel e Turismo,7.008529e+09,3001084033,7160.0,6520.0,6840.0,7440.0,6920.0,6920.0,7240.0,...,52317,48122,50739,48737,52874,56271,51096,53430,45995,51054
2,Academia Planeta Corpo,7.005369e+09,3011373971,440.0,600.0,680.0,760.0,560.0,520.0,320.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MD Predial,7.005492e+09,3001449459,960.0,920.0,1120.0,1040.0,1120.0,1240.0,1000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MD Predial,7.005492e+09,3011504476,1560.0,1280.0,1360.0,1680.0,1680.0,2920.0,1960.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,Usina Albuquerque,7.201768e+09,3014922778,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
173,Usina Fenix - BL 1,7.201768e+09,3013771324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
174,Usina Fenix - BL 3,7.201768e+09,3013939291,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175,Usina Fenix - BL 7,7.201768e+09,3013939295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
# Removendo dados desnecessários (mantendo UC)
df_excel_clean = df_excel.copy()
df_excel_clean.drop(df_excel_clean.iloc[:, 48:68], inplace=True, axis=1)
df_excel_clean.drop("Nº Cliente", inplace=True, axis=1)
df_excel_clean.drop("Cliente", inplace=True, axis=1)

df_excel_clean = df_excel_clean.head()
df_excel_clean

,UC,2019-06-01 00:00:00,2019-07-01 00:00:00,2019-08-01 00:00:00,2019-09-01 00:00:00,2019-10-01 00:00:00,2019-11-01 00:00:00,2019-12-01 00:00:00,2020-01-01 00:00:00,2020-02-01 00:00:00,...,2022-05-01 00:00:00,2022-06-01 00:00:00,2022-07-01 00:00:00,2022-08-01 00:00:00,2022-09-01 00:00:00,2022-10-01 00:00:00,2022-11-01 00:00:00,2022-12-01 00:00:00,2023-01-01 00:00:00,2023-02-01 00:00:00
0,3003858507,1246.0,1081.0,1184.0,1549.0,2045.0,4017.0,2386.0,4239.0,2854.0,...,2478.0,817.0,601.0,638.0,686.0,1216.0,1335.0,1317.0,2367.0,3056.0
1,3001084033,7160.0,6520.0,6840.0,7440.0,6920.0,6920.0,7240.0,6560.0,7440.0,...,6800.0,6760.0,6280.0,5560.0,6360.0,7440.0,7240.0,7960.0,5880.0,7560.0
2,3011373971,440.0,600.0,680.0,760.0,560.0,520.0,320.0,440.0,280.0,...,680.0,840.0,800.0,760.0,840.0,920.0,360.0,680.0,800.0,480.0
3,3001449459,960.0,920.0,1120.0,1040.0,1120.0,1240.0,1000.0,1120.0,1200.0,...,800.0,720.0,880.0,680.0,800.0,1000.0,840.0,880.0,840.0,760.0
4,3011504476,1560.0,1280.0,1360.0,1680.0,1680.0,2920.0,1960.0,2280.0,2480.0,...,1840.0,1600.0,1480.0,1440.0,1720.0,2280.0,2000.0,2160.0,2120.0,2680.0


In [49]:
# Extraindo os UCs que serão usados para filtrar os dados
selected_ucs = df_excel_clean['UC'].astype(str).tolist()
print(f"UCs selecionados do Excel: {selected_ucs}")

UCs selecionados do Excel: ['3003858507', '3001084033', '3011373971', '3001449459', '3011504476']


In [50]:
# Corrigindo os dados para uso das fórmulas (Colocando a data como índice do dataset)
df_excel_transpose1 = df_excel_clean.transpose()
df_excel_transpose = df_excel_transpose1.rename(columns=df_excel_transpose1.iloc[0])
df_excel_transpose.drop("UC", inplace=True, axis=0)
df_excel_transpose.dropna(inplace=True)
df_excel_transpose

,3003858507,3001084033,3011373971,3001449459,3011504476
2019-06-01 00:00:00,1246.0,7160.0,440.0,960.0,1560.0
2019-07-01 00:00:00,1081.0,6520.0,600.0,920.0,1280.0
2019-08-01 00:00:00,1184.0,6840.0,680.0,1120.0,1360.0
2019-09-01 00:00:00,1549.0,7440.0,760.0,1040.0,1680.0
2019-10-01 00:00:00,2045.0,6920.0,560.0,1120.0,1680.0
2019-11-01 00:00:00,4017.0,6920.0,520.0,1240.0,2920.0
2019-12-01 00:00:00,2386.0,7240.0,320.0,1000.0,1960.0
2020-01-01 00:00:00,4239.0,6560.0,440.0,1120.0,2280.0
2020-02-01 00:00:00,2854.0,7440.0,280.0,1200.0,2480.0
2020-03-01 00:00:00,3208.0,7440.0,320.0,1360.0,2240.0


In [51]:
# Transformando índice em datetime para o Excel
some_dates = np.array(df_excel_transpose.index, dtype="datetime64[D]")
idx_excel = pd.DatetimeIndex(some_dates)
df_excel_transpose.index = idx_excel

df_excel_transpose

,3003858507,3001084033,3011373971,3001449459,3011504476
2019-06-01,1246.0,7160.0,440.0,960.0,1560.0
2019-07-01,1081.0,6520.0,600.0,920.0,1280.0
2019-08-01,1184.0,6840.0,680.0,1120.0,1360.0
2019-09-01,1549.0,7440.0,760.0,1040.0,1680.0
2019-10-01,2045.0,6920.0,560.0,1120.0,1680.0
2019-11-01,4017.0,6920.0,520.0,1240.0,2920.0
2019-12-01,2386.0,7240.0,320.0,1000.0,1960.0
2020-01-01,4239.0,6560.0,440.0,1120.0,2280.0
2020-02-01,2854.0,7440.0,280.0,1200.0,2480.0
2020-03-01,3208.0,7440.0,320.0,1360.0,2240.0


In [52]:
# Convertendo Excel para formato long (igual ao CSV)
excel_long = []
for installation in df_excel_transpose.columns:
    for date in df_excel_transpose.index:
        value = df_excel_transpose.loc[date, installation]
        if pd.notna(value):  # Só adiciona valores não-nulos
            excel_long.append({
                'Date': date,
                'Installation': str(installation),  # Convertendo para string para compatibilidade
                'ConsumeValue': float(value),
                'Source': 'Excel'
            })

df_excel_long = pd.DataFrame(excel_long)
df_excel_long

,Date,Installation,ConsumeValue,Source
0,2019-06-01,3003858507,1246.0,Excel
1,2019-07-01,3003858507,1081.0,Excel
2,2019-08-01,3003858507,1184.0,Excel
3,2019-09-01,3003858507,1549.0,Excel
4,2019-10-01,3003858507,2045.0,Excel
...,...,...,...,...
220,2022-10-01,3011504476,2280.0,Excel
221,2022-11-01,3011504476,2000.0,Excel
222,2022-12-01,3011504476,2160.0,Excel
223,2023-01-01,3011504476,2120.0,Excel


In [53]:
# Lendo os dados do CSV
df_csv = pd.read_csv("../_SELECT_cb_Name_cb_Installation_fc_ForecastConsumeValue_fc_Refer_202504140945.csv")

# Convertendo a coluna Date para datetime
df_csv['Date'] = pd.to_datetime(df_csv['Date'])
df_csv['Installation'] = df_csv['Installation'].astype(str)  # Garantindo que seja string
df_csv


,Name,Installation,ConsumeValue,Date
0,DIONATH MAICON BORGES,3002029081,0,2025-05-01
1,JULIA JANSSEN PANTUZA,3010248281,0,2025-05-01
2,Filé di Gato (Bhitsol),3007358639,0,2025-05-01
3,AFERE MOTORES ELETRICOS LTDA,3006787706,0,2025-05-01
4,CLL Hotel e Turismo LTDA,3001084033,0,2025-05-01
...,...,...,...,...
7111,Condomínio Edifício Atlanta Residence,3010477511,835,2023-02-01
7112,GUILHERME PEREIRA BATISTA,3013708265,90,2023-02-01
7113,Alexander Fernandes Gonçalves,3014448834,166,2023-02-01
7114,Alex Ferreira Sampaio,3011681338,15320,2023-02-01


In [54]:
# Filtrando CSV para manter apenas os mesmos UCs do Excel
df_csv_filtered = df_csv[df_csv['Installation'].isin(selected_ucs)].copy()
df_csv_filtered['Source'] = 'CSV'
df_csv_filtered

,Name,Installation,ConsumeValue,Date,Source
4,CLL Hotel e Turismo LTDA,3001084033,0,2025-05-01,CSV
86,Lucile Confeções LTDA,3003858507,0,2025-05-01,CSV
150,ACADEMIA PLANETA CORPO LTDA,3011373971,0,2025-05-01,CSV
221,CLL Hotel e Turismo LTDA,3001084033,0,2025-04-01,CSV
388,Lucile Confeções LTDA,3003858507,0,2025-04-01,CSV
...,...,...,...,...,...
7033,ACADEMIA PLANETA CORPO LTDA,3011373971,480,2023-02-01,CSV
7064,Lucile Confeções LTDA,3003858507,3056,2023-02-01,CSV
7081,Construtora HRDominio LTDA,3011504476,2680,2023-02-01,CSV
7102,Construtora HRDominio LTDA,3001449459,760,2023-02-01,CSV


In [55]:
print(f"CSV após filtrar por UCs selecionados: {len(df_csv_filtered)} registros")
if len(df_csv_filtered) > 0:
    print(f"Período CSV filtrado: {df_csv_filtered['Date'].min()} até {df_csv_filtered['Date'].max()}")
    csv_ucs = df_csv_filtered['Installation'].unique()
    print(f"UCs encontrados no CSV: {list(csv_ucs)}")
    ucs_in_both = set(selected_ucs).intersection(set(csv_ucs))
    print(f"UCs presentes em ambos (Excel + CSV): {list(ucs_in_both)}")
else:
    print("Nenhum UC do Excel foi encontrado no CSV")

CSV após filtrar por UCs selecionados: 139 registros
Período CSV filtrado: 2023-02-01 00:00:00 até 2025-05-01 00:00:00
UCs encontrados no CSV: ['3001084033', '3003858507', '3011373971', '3011504476', '3001449459']
UCs presentes em ambos (Excel + CSV): ['3003858507', '3011373971', '3001084033', '3011504476', '3001449459']


In [56]:
print("\n=== COMBINANDO OS DADOS ===")
# Combinando os dataframes
df_combined = pd.concat([df_excel_long, df_csv_filtered], ignore_index=True)
df_combined


=== COMBINANDO OS DADOS ===


,Date,Installation,ConsumeValue,Source,Name
0,2019-06-01,3003858507,1246.0,Excel,NaN
1,2019-07-01,3003858507,1081.0,Excel,NaN
2,2019-08-01,3003858507,1184.0,Excel,NaN
3,2019-09-01,3003858507,1549.0,Excel,NaN
4,2019-10-01,3003858507,2045.0,Excel,NaN
...,...,...,...,...,...
359,2023-02-01,3011373971,480.0,CSV,ACADEMIA PLANETA CORPO LTDA
360,2023-02-01,3003858507,3056.0,CSV,Lucile Confeções LTDA
361,2023-02-01,3011504476,2680.0,CSV,Construtora HRDominio LTDA
362,2023-02-01,3001449459,760.0,CSV,Construtora HRDominio LTDA


In [57]:
# Verificando duplicatas na combinação Date + Installation
duplicate_check = df_combined.groupby(['Date', 'Installation']).size()
duplicates = duplicate_check[duplicate_check > 1]

if len(duplicates) > 0:
    print(f"\nEncontradas {len(duplicates)} combinações Date-Installation duplicadas")
    
    # Mostrando alguns exemplos das duplicatas
    print("\nExemplos das linhas duplicadas:")
    for (date, installation), count in duplicates.head(3).items():
        print(f"\nDate: {date}, Installation: {installation} ({count} ocorrências):")
        sample_dups = df_combined[(df_combined['Date'] == date) & (df_combined['Installation'] == installation)]
        print(sample_dups[['Date', 'Installation', 'ConsumeValue', 'Source']])
    
    # Estratégia para duplicatas: priorizar CSV sobre Excel (mais recente), ou somar valores
    print("\nResolvendo duplicatas...")
    
    df_combined_clean = df_combined.sort_values(['Date', 'Installation', 'Source']).drop_duplicates(
        subset=['Date', 'Installation'], keep='last')  # CSV vem depois do Excel na ordenação
    
    print(f"Dados após resolver duplicatas: {len(df_combined_clean)} registros")
else:
    df_combined_clean = df_combined.copy()
    print("Nenhuma duplicata encontrada!")


Encontradas 10 combinações Date-Installation duplicadas

Exemplos das linhas duplicadas:

Date: 2023-02-01 00:00:00, Installation: 3001084033 (2 ocorrências):
          Date Installation  ConsumeValue Source
89  2023-02-01   3001084033        7560.0  Excel
363 2023-02-01   3001084033        7560.0    CSV

Date: 2023-02-01 00:00:00, Installation: 3001449459 (2 ocorrências):
          Date Installation  ConsumeValue Source
179 2023-02-01   3001449459         760.0  Excel
362 2023-02-01   3001449459         760.0    CSV

Date: 2023-02-01 00:00:00, Installation: 3003858507 (2 ocorrências):
          Date Installation  ConsumeValue Source
44  2023-02-01   3003858507        3056.0  Excel
360 2023-02-01   3003858507        3056.0    CSV

Resolvendo duplicatas...
Dados após resolver duplicatas: 354 registros


In [58]:
# Criando informação sobre as fontes dos dados
source_info = df_combined_clean['Source'].value_counts()
print(f"\nDistribuição das fontes:")
for source, count in source_info.items():
    print(f"{source}: {count} registros")



Distribuição das fontes:
Excel: 225 registros
CSV: 129 registros


In [59]:
# Fazendo o pivot com os dados limpos
df_final = df_combined_clean.pivot(index='Date', columns='Installation', values='ConsumeValue')

# Substituindo valores NaN por 0
df_final.fillna(0, inplace=True)

# Ordenando o índice por data
df_final.sort_index(inplace=True)

# Removendo dados a partir de 2025-01-01
df_final = df_final[df_final.index < '2025-01-01']

df_final

Installation,3001084033,3001449459,3003858507,3011373971,3011504476
Date,,,,,
2019-06-01,7160.0,960.0,1246.0,440.0,1560.0
2019-07-01,6520.0,920.0,1081.0,600.0,1280.0
2019-08-01,6840.0,1120.0,1184.0,680.0,1360.0
2019-09-01,7440.0,1040.0,1549.0,760.0,1680.0
2019-10-01,6920.0,1120.0,2045.0,560.0,1680.0
...,...,...,...,...,...
2024-10-23,6460.0,1260.0,2194.0,0.0,820.0
2024-11-01,0.0,0.0,0.0,973.0,0.0
2024-11-22,6367.0,1260.0,2106.0,0.0,820.0


In [60]:
# Verificando a frequência dos dados
print("Verificando frequência dos dados:")
print(f"Primeiro registro: {df_final.index.min()}")
print(f"Último registro: {df_final.index.max()}")
print(f"Número de registros: {len(df_final)}")
print(f"Número de instalações: {len(df_final.columns)}")

Verificando frequência dos dados:
Primeiro registro: 2019-06-01 00:00:00
Último registro: 2024-12-18 00:00:00
Número de registros: 73
Número de instalações: 5


In [61]:
# Verificando se os dados são mensais
date_diffs = df_final.index.to_series().diff().dropna()
print(f"Diferenças entre datas: {date_diffs.value_counts().head()}")

# Se os dados já são mensais, tenta inferir a frequência
try:
    df_final.index.freq = pd.infer_freq(df_final.index)
    print(f"Frequência inferida: {df_final.index.freq}")
except:
    print("Não foi possível inferir frequência automaticamente")

Diferenças entre datas: Date
31 days    35
30 days    20
28 days     5
29 days     2
2 days      2
Name: count, dtype: int64
Frequência inferida: None


In [62]:
# Se a frequência não for detectada ou for diferente de mensal, resample para mensal
if df_final.index.freq != 'MS' and df_final.index.freq != 'M':
    print("Reamostrando dados para frequência mensal...")
    # Agrupa por mês e soma os valores (ou use .mean() se preferir média)
    df_final = df_final.resample('MS').mean()
    df_final.index.freq = 'MS'
    print(f"Nova frequência: {df_final.index.freq}")
    print(f"Novo shape após reamostragem: {df_final.shape}")
else:
    print("Dados já estão em frequência mensal")

Reamostrando dados para frequência mensal...
Nova frequência: <MonthBegin>
Novo shape após reamostragem: (67, 5)


In [63]:
# Verificando a estrutura dos dados
print("\n=== ESTRUTURA FINAL DOS DADOS ===")
print("Colunas disponíveis (Installations):")
print(df_final.columns.tolist()[:10])  # Mostra as primeiras 10 instalações
print(f"\nShape dos dados: {df_final.shape}")
print(f"Período dos dados: {df_final.index.min()} até {df_final.index.max()}")


=== ESTRUTURA FINAL DOS DADOS ===
Colunas disponíveis (Installations):
['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']

Shape dos dados: (67, 5)
Período dos dados: 2019-06-01 00:00:00 até 2024-12-01 00:00:00


In [64]:
# Mostrando estatísticas básicas
print("\nEstatísticas básicas:")
non_zero_counts = (df_final > 0).sum()
print(f"Instalações com dados não-zero: {(non_zero_counts > 0).sum()}")
print(f"Média de registros por instalação: {non_zero_counts.mean():.1f}")


Estatísticas básicas:
Instalações com dados não-zero: 5
Média de registros por instalação: 64.8


In [65]:
df_final

Installation,3001084033,3001449459,3003858507,3011373971,3011504476
Date,,,,,
2019-06-01,7160.0,960.0,1246.0,440.0,1560.0
2019-07-01,6520.0,920.0,1081.0,600.0,1280.0
2019-08-01,6840.0,1120.0,1184.0,680.0,1360.0
2019-09-01,7440.0,1040.0,1549.0,760.0,1680.0
2019-10-01,6920.0,1120.0,2045.0,560.0,1680.0
...,...,...,...,...,...
2024-08-01,3230.0,630.0,1037.0,573.5,410.0
2024-09-01,3090.0,630.0,1025.0,593.5,410.0
2024-10-01,3230.0,630.0,1097.0,560.0,410.0


In [66]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from statsmodels.tools.eval_measures import rmse
import itertools
from datetime import datetime
import os

# ============================================================================
# CONFIGURAÇÃO - MUDE ESTE VALOR PARA CADA ARQUIVO
# ============================================================================
INSTALLATION_ID = "3001449459"  # MUDE PARA: 3001449459, 3003858507, 3011373971, 3011504476
# ============================================================================

print(f"{'='*80}")
print(f"SARIMA GRID SEARCH - Installation {INSTALLATION_ID}")
print(f"{'='*80}\n")

# Carregue seu df_final aqui (copie o código de preparação dos dados)
# Por exemplo:
# df_final = pd.read_csv("seu_arquivo_preparado.csv", index_col='Date', parse_dates=True)
# OU execute todo o código de preparação aqui

# Definindo os ranges dos hiperparâmetros
p_range = range(0, 11)  # 0-10
d_range = range(0, 2)   # 0-1
q_range = range(0, 11)  # 0-10
P_range = range(0, 11)  # 0-10 (seasonal)
D_range = range(0, 2)   # 0-1 (seasonal)
Q_range = range(0, 11)  # 0-10 (seasonal)
s = 12  # sazonalidade fixa

# Gerando todas as combinações possíveis
order_combinations = list(itertools.product(p_range, d_range, q_range))
seasonal_order_combinations = list(itertools.product(P_range, D_range, Q_range))

total_combinations = len(order_combinations) * len(seasonal_order_combinations)
print(f"Total de combinações: {total_combinations}\n")

# Nome do arquivo de resultados
results_file = f"sarima_results_{INSTALLATION_ID}.csv"

# Inicializa ou carrega arquivo existente
if os.path.exists(results_file):
    print(f"Arquivo {results_file} encontrado.")
    existing_results = pd.read_csv(results_file)
    print(f"Resultados anteriores: {len(existing_results)} registros\n")
    
    user_input = input("Deseja CONTINUAR de onde parou (c) ou RECOMEÇAR do zero (r)? [c/r]: ").strip().lower()
    if user_input == 'r':
        print("Recomeçando do zero...")
        existing_results = pd.DataFrame(columns=[
            'Installation', 'p', 'd', 'q', 'P', 'D', 'Q', 's',
            'MAPE', 'MAE', 'WMAPE', 'RMSE', 
            'Status', 'Error_Message', 'Timestamp'
        ])
        existing_results.to_csv(results_file, index=False)
        print("Arquivo resetado!\n")
else:
    print(f"Criando novo arquivo {results_file}...")
    existing_results = pd.DataFrame(columns=[
        'Installation', 'p', 'd', 'q', 'P', 'D', 'Q', 's',
        'MAPE', 'MAE', 'WMAPE', 'RMSE', 
        'Status', 'Error_Message', 'Timestamp'
    ])
    existing_results.to_csv(results_file, index=False)
    print("Arquivo criado!\n")

# Recarrega para verificação
existing_results = pd.read_csv(results_file)

# Função para verificar se uma combinação já foi testada
def already_tested(p, d, q, P, D, Q):
    if len(existing_results) == 0:
        return False
    mask = (
        (existing_results['p'] == p) &
        (existing_results['d'] == d) &
        (existing_results['q'] == q) &
        (existing_results['P'] == P) &
        (existing_results['D'] == D) &
        (existing_results['Q'] == Q)
    )
    return mask.any()

# Função para salvar resultado
def save_result(p, d, q, P, D, Q, mape, mae, wmape, rmse_val, status, error_msg=""):
    result = {
        'Installation': INSTALLATION_ID,
        'p': p, 'd': d, 'q': q,
        'P': P, 'D': D, 'Q': Q, 's': s,
        'MAPE': mape,
        'MAE': mae,
        'WMAPE': wmape,
        'RMSE': rmse_val,
        'Status': status,
        'Error_Message': error_msg,
        'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    result_df = pd.DataFrame([result])
    result_df.to_csv(results_file, mode='a', header=False, index=False)

# Verifica se a coluna existe
if INSTALLATION_ID not in df_final.columns:
    print(f"ERRO: Installation {INSTALLATION_ID} não encontrada no dataframe!")
    print(f"Colunas disponíveis: {df_final.columns.tolist()}")
    exit()

# Prepara os dados
print("Preparando dados...")
data = df_final[INSTALLATION_ID]
end = len(data) - 1
half = data.iloc[23:]

print(f"Dados preparados: {len(data)} registros")
print(f"Período: {data.index.min()} até {data.index.max()}\n")

# Contadores
combination_count = 0
skipped_count = 0
success_count = 0
error_count = 0

start_time = datetime.now()
last_save_batch_time = datetime.now()
results_buffer = []

print(f"{'='*80}")
print(f"Início do processamento: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}\n")

# Loop principal
for order in order_combinations:
    p, d, q = order
    
    for seasonal_order in seasonal_order_combinations:
        P, D, Q = seasonal_order
        
        combination_count += 1
        
        # Verifica se já foi testado
        if already_tested(p, d, q, P, D, Q):
            skipped_count += 1
            continue
        
        # Progress update a cada 50 combinações
        if combination_count % 50 == 0:
            elapsed = (datetime.now() - start_time).total_seconds()
            rate = combination_count / elapsed if elapsed > 0 else 0
            remaining = (total_combinations - combination_count) / rate if rate > 0 else 0
            
            print(f"Progresso: {combination_count}/{total_combinations} ({combination_count/total_combinations*100:.1f}%)")
            print(f"  order=({p},{d},{q}), seasonal_order=({P},{D},{Q},12)")
            print(f"  Sucesso: {success_count} | Erros: {error_count} | Pulados: {skipped_count}")
            print(f"  Buffer: {len(results_buffer)} | Taxa: {rate:.2f}/s | ETA: {remaining/60:.1f} min")
            print()
        
        try:
            # Treina o modelo
            model = SARIMAX(
                data,
                order=(p, d, q),
                seasonal_order=(P, D, Q, s),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            
            results = model.fit(disp=False, maxiter=100, method='lbfgs')
            
            # Faz as predições
            predictions = results.predict(start=0, end=end, dynamic=False, typ="levels")
            half_pred = predictions.iloc[23:]
            
            # Calcula as métricas
            mape = mean_absolute_percentage_error(half, half_pred)
            mae = mean_absolute_error(half, half_pred)
            wmape = mean_absolute_percentage_error(half, half_pred, sample_weight=half)
            rmse_val = rmse(half, half_pred)
            
            # Adiciona ao buffer
            results_buffer.append({
                'Installation': INSTALLATION_ID,
                'p': p, 'd': d, 'q': q,
                'P': P, 'D': D, 'Q': Q, 's': s,
                'MAPE': mape,
                'MAE': mae,
                'WMAPE': wmape,
                'RMSE': rmse_val,
                'Status': "Success",
                'Error_Message': "",
                'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            success_count += 1
            
        except Exception as e:
            error_count += 1
            error_msg = str(e)[:200]
            
            # Adiciona ao buffer
            results_buffer.append({
                'Installation': INSTALLATION_ID,
                'p': p, 'd': d, 'q': q,
                'P': P, 'D': D, 'Q': Q, 's': s,
                'MAPE': np.nan,
                'MAE': np.nan,
                'WMAPE': np.nan,
                'RMSE': np.nan,
                'Status': "Error",
                'Error_Message': error_msg,
                'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
        
        # Salva a cada 50 resultados ou a cada 2 minutos
        current_time = datetime.now()
        if len(results_buffer) >= 50 or (current_time - last_save_batch_time).total_seconds() > 120:
            if results_buffer:
                try:
                    buffer_df = pd.DataFrame(results_buffer)
                    buffer_df.to_csv(results_file, mode='a', header=False, index=False)
                    print(f"✓ Salvou {len(results_buffer)} resultados no arquivo")
                    results_buffer = []
                    last_save_batch_time = current_time
                except Exception as e:
                    print(f"✗ ERRO ao salvar: {e}")

# Salva resultados restantes
if results_buffer:
    try:
        buffer_df = pd.DataFrame(results_buffer)
        buffer_df.to_csv(results_file, mode='a', header=False, index=False)
        print(f"✓ Salvou {len(results_buffer)} resultados finais")
    except Exception as e:
        print(f"✗ ERRO ao salvar resultados finais: {e}")

# Resumo final
end_time = datetime.now()
elapsed = (end_time - start_time).total_seconds()

print(f"\n{'='*80}")
print(f"PROCESSAMENTO CONCLUÍDO!")
print(f"{'='*80}")
print(f"Installation: {INSTALLATION_ID}")
print(f"Tempo total: {elapsed/3600:.2f} horas ({elapsed/60:.2f} minutos)")
print(f"Arquivo: {results_file}")
print(f"Total testado: {combination_count}")
print(f"Pulados: {skipped_count}")
print(f"Sucessos: {success_count}")
print(f"Erros: {error_count}")
print(f"{'='*80}\n")

# Mostra os melhores resultados
print("Carregando e analisando resultados...")
final_results = pd.read_csv(results_file)
successful = final_results[final_results['Status'] == 'Success'].copy()

if len(successful) > 0:
    print(f"\nTotal de testes bem-sucedidos: {len(successful)}")
    
    # Ordena por MAPE
    successful = successful.sort_values('MAPE')
    
    print("\n" + "="*80)
    print(f"TOP 10 MODELOS (menor MAPE) - Installation {INSTALLATION_ID}")
    print("="*80)
    print(successful[['p', 'd', 'q', 'P', 'D', 'Q', 'MAPE', 'MAE', 'WMAPE', 'RMSE']].head(10).to_string(index=False))
    
    best = successful.iloc[0]
    print(f"\n{'='*80}")
    print("MELHOR MODELO:")
    print(f"{'='*80}")
    print(f"order=({int(best['p'])}, {int(best['d'])}, {int(best['q'])})")
    print(f"seasonal_order=({int(best['P'])}, {int(best['D'])}, {int(best['Q'])}, 12)")
    print(f"MAPE: {best['MAPE']*100:.2f}%")
    print(f"MAE: {best['MAE']:.2f}")
    print(f"WMAPE: {best['WMAPE']*100:.2f}%")
    print(f"RMSE: {best['RMSE']:.2f}")
    print(f"{'='*80}")
else:
    print("\nNenhum teste bem-sucedido encontrado.")

print(f"\nResultados completos salvos em: {results_file}")

SARIMA GRID SEARCH - Installation 3001449459

Total de combinações: 58564

Arquivo sarima_results_3001449459.csv encontrado.
Resultados anteriores: 57117 registros

Preparando dados...
Dados preparados: 67 registros
Período: 2019-06-01 00:00:00 até 2024-12-01 00:00:00

Início do processamento: 2025-10-16 09:54:18

✓ Salvou 19 resultados no arquivo
✓ Salvou 13 resultados no arquivo
Progresso: 57150/58564 (97.6%)
  order=(10,1,5), seasonal_order=(1,1,4,12)
  Sucesso: 32 | Erros: 0 | Pulados: 57117
  Buffer: 0 | Taxa: 225.86/s | ETA: 0.1 min

✓ Salvou 19 resultados no arquivo
✓ Salvou 11 resultados no arquivo
✓ Salvou 7 resultados no arquivo
✓ Salvou 7 resultados no arquivo
Progresso: 57200/58564 (97.7%)
  order=(10,1,5), seasonal_order=(3,1,10,12)
  Sucesso: 82 | Erros: 0 | Pulados: 57117
  Buffer: 6 | Taxa: 72.41/s | ETA: 0.3 min

✓ Salvou 11 resultados no arquivo
✓ Salvou 38 resultados no arquivo
Progresso: 57250/58564 (97.8%)
  order=(10,1,5), seasonal_order=(6,0,5,12)
  Sucesso: 132 